# Reconstruct images with the current autoencoder


The current model in the historical `pvcracks.vae` namespace is
`UNetAutoencoder3CHPlus`, a deterministic skip-connected autoencoder. It
reconstructs **crack, busbar, dark** binary masks in that order, with shape
`[N, 3, 400, 400]`, and produces 320 pooled bottleneck values. It needs
encoder skip features to reconstruct an image; random latent decoding is
unavailable.

Install the runtime from this checkout with `python -m pip install -e .`.
The matching checkpoint and all three XGBoost models ship with the package.
These examples use the 26 module 209 masks successfully segmented by the
supplied FinalVAE3CH analysis. No Box folder or training package is required.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pvcracks.vae import load_unet_ae_crack_plus
from pvcracks.vae.features import CHANNELS, extract_features
from pvcracks.vae.inference import reconstruct_masks, reconstruction_scores

# Works from the repository root, docs/Examples, or a notebook subdirectory.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "docs/data/autoencoder_3ch_module_209.npz").is_file())
DATA = ROOT / "docs/data"
DEVICE = "cpu"  # Set to "cuda" if desired.
BATCH_SIZE = 4
masks = np.load(DATA / "autoencoder_3ch_module_209.npz")["masks"]
metadata = pd.read_csv(DATA / "autoencoder_3ch_module_209.csv")
model, checkpoint = load_unet_ae_crack_plus(device=DEVICE)
print(f"{len(masks)} masks; input {masks.shape[1:]}; embedding {model.latent_dim}")

In [ ]:
probabilities = reconstruct_masks(model, masks[:3], batch_size=BATCH_SIZE)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row in range(3):
    axes[row, 0].imshow(masks[row].transpose(1, 2, 0))
    axes[row, 1].imshow(probabilities[row].numpy().transpose(1, 2, 0))
    axes[row, 2].imshow(masks[row, 0], cmap="gray")
    for ax in axes[row]:
        ax.set_axis_off()
for ax, title in zip(axes[0], ["Input masks (RGB)", "Reconstruction probabilities", "Input crack"]):
    ax.set_title(title)
fig.tight_layout()
plt.show()

This replaces the previous random-image generation example. A 320-value
bottleneck vector alone cannot recover this model's encoder skip maps.
Use `model.encode(masks)` for analysis and `reconstruct_masks(model,
masks)` for reconstructions.